[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Реализуйте **beam search** — поиск последовательности, который на каждом шаге хранит несколько лучших гипотез. В отличие от greedy decoding, beam search может временно сохранить не самый лучший следующий token, если вся продолженная последовательность окажется лучше.

### Score последовательности
`log_prob_fn(tokens)` возвращает log-probabilities следующего token. Score гипотезы — сумма log-probabilities её переходов. Сумма используется потому, что логарифм превращает произведение вероятностей в сложение и избегает численного underflow. Чем score ближе к нулю, тем гипотеза лучше.

### Живые и завершённые beams
Beam удобно хранить как пару `(score, token_list)`. Гипотеза, закончившаяся `eos_token`, уже завершена: её нельзя расширять новыми token, но нужно сохранять среди кандидатов. Остановка безопасна, когда все оставшиеся лучшие beams завершены, либо достигнут `max_len`.

### Почему beam может победить greedy
Greedy с `beam_width=1` выбирает лучший token локально. При `beam_width>1` алгоритм сравнивает несколько путей после следующего расширения, поэтому путь со слегка худшим первым token может получить намного лучшее продолжение.

### Контракт
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### План реализации
1. Начните с `[(0.0, [start_token])]`.
2. На шаге расширьте каждую незавершённую гипотезу наиболее вероятными next tokens.
3. Прибавьте log-probability token к накопленному score.
4. Отсортируйте общий набор кандидатов и оставьте `beam_width` лучших.
5. Остановитесь при завершении beams или достижении `max_len`, затем верните sequence лучшего score.

### Быстрые самопроверки
- результат — list, начинающийся с `start_token`;
- при `beam_width=1` поведение совпадает с greedy;
- token после EOS никогда не добавляется;
- искусственный пример с отложенной выгодой находится beam search, но не greedy.

### Типичные ошибки
- сортировка log-probabilities в неправильном направлении;
- сравнение только next-token score без накопленного score;
- повторное расширение завершённой EOS-гипотезы;
- pruning отдельно внутри каждого beam вместо глобального pruning кандидатов;
- возврат score вместо списка token.

### Официальные материалы и примеры
- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html) — выбор лучших token-кандидатов;
- [Seq2Seq translation tutorial](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html) — autoregressive decoding loop, от которого beam search отличается хранением нескольких гипотез.

### Вопросы для самопроверки
1. Почему scores удобно хранить в log-space?
2. Где именно beam search делает глобальный выбор кандидатов?
3. Почему без length normalization короткие последовательности часто получают преимущество?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    pass  # maintain beams, expand, prune, return best

In [ ]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')